# 01 · Customer Segmentation (RFM + K-Means)

**Business question:** *Which types of customers do we have, and how should we treat each group?*

**Approach**

1. Load one row per customer from the SQL view `dbo.vw_CustomerFeatures`.
2. Transform and scale the features (money and frequency are heavily skewed).
3. Group customers with **K-Means**, choosing K with the elbow method, the silhouette score and business interpretability.
4. Name and profile each segment, visualise them, and turn them into recommended actions.

**Features (RFM + extras)**

| Feature | Meaning |
|---|---|
| `Recency` | Days since the customer's last purchase (lower = more recent) |
| `Frequency` | Number of distinct orders |
| `Monetary` | Total revenue from the customer |
| `TotalQuantity` | Total units bought |
| `UniqueProducts` | Number of different products bought |
| `AOV` | Average order value = Monetary / Frequency |

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Find the project root (the folder that contains "src"), whatever the working directory is
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "db.py").exists()), None)
if root is None:
    raise FileNotFoundError("Open this notebook from inside the retail-intelligence folder (it needs the src/ folder next to it).")
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.db import read_sql       # runs a SELECT on SQL Server and returns a DataFrame

plt.rcParams["figure.dpi"] = 100

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## 2. Load the data

In [ ]:
df_customer = read_sql("SELECT * FROM dbo.vw_CustomerFeatures")

print("Customers:", df_customer.shape[0])
df_customer.head()

Quick quality check: there should be no missing values, and the summary statistics show how skewed the data is (the mean is far above the median for money and quantity).

In [ ]:
display(df_customer.isnull().sum().to_frame("missing values").T)
df_customer.describe().T

## 3. Prepare the features

* `log1p` (log of 1 + x) shrinks the long right tail, so a few very large customers do not dominate the clusters.
* `StandardScaler` puts every feature on the same scale, because K-Means is distance-based.

In [ ]:
FEATURES = ["Recency", "Frequency", "Monetary", "TotalQuantity", "UniqueProducts", "AOV"]

X_log = np.log1p(df_customer[FEATURES])
X_scaled = StandardScaler().fit_transform(X_log)

## 4. Choose the number of clusters (K)

In [ ]:
k_values = range(2, 9)
rows = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    rows.append({
        "K": k,
        "Inertia": model.inertia_,
        "Silhouette": silhouette_score(X_scaled, labels),
    })

k_results = pd.DataFrame(rows)
display(k_results)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(k_results["K"], k_results["Inertia"], marker="o")
axes[0].set(title="Elbow method", xlabel="Number of clusters (K)", ylabel="Inertia")

axes[1].plot(k_results["K"], k_results["Silhouette"], marker="o", color="tab:orange")
axes[1].set(title="Silhouette score", xlabel="Number of clusters (K)", ylabel="Silhouette")

plt.tight_layout()
plt.show()

**Decision: K = 4**

* The silhouette score is highest at K = 2, but two clusters only separate *active* from *inactive* customers, which is too coarse to act on.
* The elbow curve bends gradually (no sharp corner), and the biggest drops in inertia happen before K = 4. K = 4 gives four groups that are clearly different and easy to describe to a marketing team.
* The trade-off is a lower silhouette score (the segments overlap somewhat, which is normal for behavioural data). It is a business-driven choice, not a purely statistical one.

## 5. Fit the final model and profile the clusters

In [ ]:
N_CLUSTERS = 4

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
df_customer["Cluster"] = kmeans.fit_predict(X_scaled)

# The median is used for profiling because it is not distorted by extreme customers.
cluster_median = df_customer.groupby("Cluster")[FEATURES].median()
cluster_median

### Name the segments

Cluster numbers (0, 1, 2, 3) are arbitrary, so names are assigned **from the profile**, not hard-coded to a number:

1. The cluster with the highest median spend = **Active High-Value**.
2. Of the remaining three, the two that have been silent the longest are **Inactive** (the one that spends more is *Low-Mid*, the other *Very-Low*).
3. The last one is **Recent Mid-Value**.

In [ ]:
def name_segments(profile: pd.DataFrame) -> dict:
    """Map each cluster id to a business-friendly segment name using its median profile."""
    best = profile["Monetary"].idxmax()                      # biggest spenders
    rest = profile.drop(index=best)

    inactive = rest["Recency"].nlargest(2).index             # the two longest-silent clusters
    recent = rest.drop(index=inactive).index[0]              # the remaining, recently active one

    low_mid = rest.loc[inactive, "Monetary"].idxmax()
    very_low = rest.loc[inactive, "Monetary"].idxmin()

    return {
        best: "Active High-Value",
        recent: "Recent Mid-Value",
        low_mid: "Inactive Low-Mid-Value",
        very_low: "Inactive Very-Low-Value",
    }


SEGMENT_ORDER = ["Active High-Value", "Recent Mid-Value",
                 "Inactive Low-Mid-Value", "Inactive Very-Low-Value"]

segment_names = name_segments(cluster_median)
df_customer["Segment"] = df_customer["Cluster"].map(segment_names)

segment_profile = (
    df_customer.groupby("Segment")
    .agg(
        Customers=("CustomerID", "count"),
        MedianRecency=("Recency", "median"),
        MedianFrequency=("Frequency", "median"),
        MedianMonetary=("Monetary", "median"),
        MedianQuantity=("TotalQuantity", "median"),
        MedianUniqueProducts=("UniqueProducts", "median"),
        MedianAOV=("AOV", "median"),
    )
    .reindex(SEGMENT_ORDER)
)
segment_profile.insert(1, "Share_%", segment_profile["Customers"] / len(df_customer) * 100)
segment_profile

## 6. Visualise the segments

### 6.1 How big is each segment?

In [ ]:
counts = df_customer["Segment"].value_counts().reindex(SEGMENT_ORDER)

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(counts.index, counts.values, color=["#2e7d32", "#1976d2", "#f9a825", "#c62828"])

for bar, value in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, value, f"{value:,}\n({value / len(df_customer):.1%})",
            ha="center", va="bottom", fontsize=9)

ax.set(title="Customers per segment", ylabel="Number of customers")
ax.set_ylim(0, counts.max() * 1.18)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### 6.2 Where do the segments sit?  (log scales, because the data is skewed)

In [ ]:
colors = dict(zip(SEGMENT_ORDER, ["#2e7d32", "#1976d2", "#f9a825", "#c62828"]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for segment in SEGMENT_ORDER:
    subset = df_customer[df_customer["Segment"] == segment]
    axes[0].scatter(subset["Recency"], subset["Monetary"], alpha=0.45, s=14, label=segment, color=colors[segment])
    axes[1].scatter(subset["Frequency"], subset["Monetary"], alpha=0.45, s=14, label=segment, color=colors[segment])

axes[0].set(title="Recency vs Monetary", xlabel="Recency (days)", ylabel="Monetary (log scale)", yscale="log")
axes[1].set(title="Frequency vs Monetary", xlabel="Frequency (log scale)", ylabel="Monetary (log scale)",
            xscale="log", yscale="log")
axes[0].legend()

plt.tight_layout()
plt.show()

### 6.3 Segment fingerprint

Each feature is rescaled to 0–1 *across the four segments* (0 = lowest segment, 1 = highest), so the heat-map shows what makes each segment different.

In [ ]:
medians = df_customer.groupby("Segment")[FEATURES].median().reindex(SEGMENT_ORDER)
normalised = (medians - medians.min()) / (medians.max() - medians.min())

fig, ax = plt.subplots(figsize=(9, 4))
image = ax.imshow(normalised.values, cmap="YlGnBu", aspect="auto", vmin=0, vmax=1)

ax.set_xticks(range(len(FEATURES)), FEATURES, rotation=20)
ax.set_yticks(range(len(SEGMENT_ORDER)), SEGMENT_ORDER)

for i in range(normalised.shape[0]):
    for j in range(normalised.shape[1]):
        value = normalised.values[i, j]
        ax.text(j, i, f"{value:.2f}", ha="center", va="center",
                color="white" if value > 0.6 else "black", fontsize=9)

ax.set_title("Normalised median profile per segment")
fig.colorbar(image, ax=ax, label="0 = lowest segment, 1 = highest segment")
plt.tight_layout()
plt.show()

## 7. From segments to actions

| Segment | Who they are | Recommended action *(proposal, not yet tested)* |
|---|---|---|
| **Active High-Value** | Buy often, spend the most, bought recently | Protect them: loyalty / VIP terms, early access, dedicated account management |
| **Recent Mid-Value** | Bought recently, moderate spend and basket | Grow them: cross-sell and bundle offers to raise order value and frequency |
| **Inactive Low-Mid-Value** | Decent past spend but silent for about a year | Win them back: targeted reactivation campaign with a personalised offer |
| **Inactive Very-Low-Value** | Few, small orders and silent for about a year | Low-cost automated emails only; do not invest heavy marketing budget |

The labelled table below can be joined back to SQL Server or loaded into a BI tool.

In [ ]:
customer_segments = df_customer[["CustomerID", "Segment"]]
customer_segments.head()

## 8. Export for the web app

The Streamlit app in `app/` does not connect to SQL Server. It reads small files saved here, so it can run anywhere (Docker, Streamlit Community Cloud). Re-run this cell whenever the segments change.

In [ ]:
ARTIFACTS = root / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

df_customer[["CustomerID", "Segment"] + FEATURES].to_csv(ARTIFACTS / "customer_segments.csv", index=False)
segment_profile.reset_index().to_csv(ARTIFACTS / "segment_profile.csv", index=False)

print("Saved:", sorted(f.name for f in ARTIFACTS.glob("*")))